# Lab 1: Text Analytics for Business – Getting Started

**Business question:** Which customer messages may still need staff help, and what does the text show that a star rating does not?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | 24 customer messages | about 35 min | Code outputs, decision note and checkpoint answer |
| 2 Practice | CFPB complaint extract | about 30–45 min | Three exercises and a short interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

**How to run a cell:** click inside a code cell and press Shift+Enter. A number appears to the left when the cell has run. Markdown cells like this one hold instructions; they do not run Python.

## Part 1 Guided core

### 1.1 Choose your Python kernel

The **kernel** is the Python environment that runs your code cells. Click the kernel picker at the top right of the notebook and choose `ban6403`. (In GitHub Codespaces, choose the Python 3.13 kernel.)

Is `ban6403` missing from the list? Then your setup is not finished yet. Go back to the setup steps in **Module 0**, then return here.

### 1.2 Import pandas

`pandas` is the Python library we use to read and inspect tables. The cell below imports it once and calls it `pd`. No output means it worked. If you get `ModuleNotFoundError`, check that `ban6403` is selected (1.1). If it is, pandas is missing from that environment: run the Module 0 readiness check.

In [ ]:
import pandas as pd

### 1.3 Point to the data folder

This notebook runs from the `notebooks` folder of your repository. The data sits next to it, in the `data` folder. In a file path, `..` means "one folder up". So `../data` is the data folder, and the same path works on every computer.

`Path` turns text into a file path that works on Windows, macOS and Linux. The `/` operator joins a folder and a file name into one path. The cell stores the data folder as `DATA`. Then it builds the path of today's file, `data/raw/customer_feedback_m01.csv`, and shows it. On Windows the output starts with `WindowsPath` instead of `PosixPath`.

In [ ]:
from pathlib import Path

# ".." means one folder up, so this is the data folder next to notebooks/.
DATA = Path("../data")

# The / operator joins folder names and a file name into one path.
FEEDBACK_FILE = DATA / "raw" / "customer_feedback_m01.csv"
FEEDBACK_FILE

### 1.4 Load the feedback messages

The file holds 24 short customer messages that we wrote for this course. They describe no real customers. A data dictionary sits beside the file in `data/raw`.

`read_csv` loads the file into a table called `feedback`. One message is empty. `keep_default_na=False` keeps it as empty text instead of `NaN` (pandas' marker for a missing value), so we can still see it later. This cell shows no output. That is normal.

In [ ]:
# keep_default_na=False keeps the empty message as "" instead of NaN.
feedback = pd.read_csv(FEEDBACK_FILE, keep_default_na=False)

### 1.5 Look at the first five rows

A **DataFrame** is a pandas table with named columns. `head()` shows the first five rows and does not change the data. Read the messages, not only the star ratings.

In [ ]:
# Show the first five rows; head() does not change the table.
feedback.head()

### 1.6 List the column names

You should see five names: `feedback_id`, `channel`, `product`, `stars` and `text`. The `feedback_id` column links each message to every later output, so we always keep it.

In [ ]:
# The column names of the table.
feedback.columns

### 1.7 Check the table size

`shape` gives (rows, columns). You should see `(24, 5)`. This tells you the file loaded completely. It does not tell you whether the data can answer the business question.

In [ ]:
# (rows, columns)
feedback.shape

### 1.8 Keep the columns we will read

A list of column names selects a smaller table. We keep the ID, the rating and the text for the first five messages. This is the preview you will save in 1.11.

In [ ]:
# A list of column names inside the brackets keeps only those columns.
# head(5) then keeps the first five rows.
preview = feedback[["feedback_id", "stars", "text"]].head(5)
preview

### 1.9 Read one complete message

A table may cut long text short on screen. This cell prints FB001 in full. `loc` selects the row where `feedback_id` equals `"FB001"`.

FB001 has five stars, but the charger is missing. Keep this message in mind for your decision note in 1.13.

In [ ]:
# The condition is True only for the row whose ID is FB001.
# .loc[condition, "text"] keeps that row and the text column.
message = feedback.loc[feedback["feedback_id"] == "FB001", "text"]

# message is still a column with one value; iloc[0] takes that first value.
print(message.iloc[0])

### 1.10 Make an output folder

We save new files in `outputs/m01`, apart from the raw data. Like `../data`, the path starts one folder up. `exist_ok=True` means the cell also works when you run it again. This cell shows no output.

In [ ]:
OUTPUT_DIR = Path("../outputs/m01")
# parents=True also creates outputs/ if it does not exist yet; exist_ok=True accepts an existing folder.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.11 Save the preview as a CSV file

`to_csv` writes the five-row preview from 1.8. `index=False` leaves out pandas' row numbers. The cell shows the path of the new file. Open it in VS Code's file list and check that it holds FB001 to FB005.

In [ ]:
# As in 1.3, the / operator joins a folder and a file name into one path.
PREVIEW_PATH = OUTPUT_DIR / "first_feedback_preview.csv"
preview.to_csv(PREVIEW_PATH, index=False)
PREVIEW_PATH

### 1.12 Confirm that the file exists

You should see `True`. The raw file `data/raw/customer_feedback_m01.csv` is not changed by anything in this lab.

In [ ]:
# exists() is True when a file is saved at this path.
PREVIEW_PATH.exists()

### 1.13 Your decision note

Imagine you work with a service manager. Each day, the team can read only about ten messages, so the manager wants a list of messages that may still need staff help.

Write 4–6 sentences in the cell below. Double-click the cell to edit it, then press Shift+Enter to show it formatted. There is no single right answer: a short, honest answer to each of the four prompts is enough.

1. Who uses this list, and what do they decide?
2. FB001 has five stars. What does its text tell you that the rating does not?
3. Name one method that could help draft the list, for example a keyword rule. How could it get FB001 or another message wrong?
4. What would show that the list helps? Name one limit of our 24 made-up messages.

**Your answer:**

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. Which file did you save in 1.11, and which feedback IDs does it contain?
2. Why do we write new files to `outputs/m01` and never change `data/raw/customer_feedback_m01.csv`?

**Your answer:**



## Part 2 Practice: real complaints

Now look at real customer text. The course extract holds 10,000 complaints that consumers sent to the U.S. Consumer Financial Protection Bureau (CFPB) in the second half of 2025. Each row is one complaint, written by the consumer. Details are in `data/course/README.md`.

Things to know about this data:

- These are real people. CFPB replaces personal details with `XXXX`. Hidden dates look like `XX/XX/XXXX` or `XX/XX/year>`; Module 3 looks at these marks closely. Do not try to find out who wrote a complaint.
- Only people who chose to complain are here, and we fixed the number of rows per product. So the counts do not show how common each problem is.
- The `split` column divides the rows into `train`, `valid` and `test`. In this course we explore only `train`. The other two are kept aside so later models can be checked on complaints they have not seen.

### 2.1 Load the complaint extract

You should see `(10000, 13)`: 10,000 complaints and 13 columns.

In [ ]:
# Read the complaints; shape gives (rows, columns).
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
complaints.shape

### 2.2 List the columns

`complaint_id` plays the same role as `feedback_id`: it links every output back to the original complaint. `narrative` is the complaint text. `product` and `issue` are categories the consumer chose.

In [ ]:
# The column names of the complaint table.
complaints.columns

### 2.3 Count the splits and the products

The first cell counts rows per split, the second rows per product. Expect 6,000 train, 2,000 valid and 2,000 test, and six products.

In [ ]:
# Count how many complaints belong to each split.
complaints["split"].value_counts()

In [ ]:
# Count how many complaints belong to each product.
complaints["product"].value_counts()

### 2.4 Keep the training complaints

`eq("train")` marks the rows whose split is `train`, and the outer brackets keep only those rows. Expect `(6000, 13)`. We use `train` for all practice work below.

In [ ]:
# eq("train") gives True for training rows; the outer brackets keep those rows.
train = complaints[complaints["split"].eq("train")]
train.shape

### 2.5 How long are the complaints?

`str.len()` counts the characters in each narrative, and `describe()` summarizes those counts. `count` is the number of complaints, `mean` the average length and `std` the standard deviation (how far lengths typically are from the mean). `min` and `max` are the shortest and longest. The `50%` row is the median complaint length: half are shorter and half are longer. `25%` and `75%` mark the middle half. `.round()` shows the summary rounded to whole numbers.

The second cell prints the length of the longest message from Part 1 for comparison. Compare it with the `50%` row: a typical complaint is many times longer than any of those 24 messages.

In [ ]:
# Characters per complaint, then a summary of those numbers.
lengths = train["narrative"].str.len()
lengths.describe().round()

In [ ]:
# .str.len() counts the characters in each message; .max() takes the largest count.
print("longest message in Part 1:", feedback["text"].str.len().max(), "characters")

### 2.6 Pick five complaints to read

We chose five training complaints, one from each of five products, each short enough to read in a minute. We select them by `complaint_id`, just as we selected FB001 by its ID. `set_index` makes the IDs the row labels, so `.loc` can look rows up by ID. The table shows what the consumer chose as product and issue, and which company the complaint is about.

In [ ]:
import textwrap

reading_ids = [14438857, 14412986, 14413024, 14387677, 14620078]

# Use complaint_id as the row label, so we can look rows up by ID.
train_by_id = train.set_index("complaint_id")

# .loc[list of IDs, list of columns] keeps those rows in that order.
reading = train_by_id.loc[reading_ids, ["product", "issue", "company", "narrative"]]

# Show the three short columns; the narratives are printed in 2.7.
reading[["product", "issue", "company"]]

### 2.7 Read the five complaints in full

The loop prints each complaint in turn: its ID, product and issue, then the full narrative. `textwrap.fill` only breaks long lines so they fit on screen; no words are removed.

As you read, ask two questions for each complaint: **Who at the company would act on it?** and **What would they decide?** You will answer them for two complaints in the Practice interpretation.

In [ ]:
# iterrows() gives one row at a time: its label (the ID) and its values.
for complaint_id, row in reading.iterrows():
    print(complaint_id, "|", row["product"], "|", row["issue"])
    # Break the narrative into lines of at most 100 characters.
    print(textwrap.fill(row["narrative"], width=100))
    print()

### Exercise 1: read a complaint of your choice

Choose one complaint from `train` that is not one of the five above, for example one about credit reporting. You can find IDs with `train[["complaint_id", "product", "issue"]].head(20)`.

Select it by its `complaint_id` and print its product, issue, company and full narrative. Note that `complaint_id` is a number, so write it without quotation marks.

In [ ]:
# Exercise 1: your code here
# TODO: store the complaint_id you chose in a variable, for example my_id = 14414866 (no quotation marks).
# TODO: keep the row of train whose complaint_id equals my_id.
# TODO: print its product, issue and company, then its full narrative (textwrap.fill helps).

### Exercise 2: count complaints per product in `train`

Count how many training complaints each product has. Compare your result with the counts for the whole extract in 2.3.

In [ ]:
# Exercise 2: your code here
# TODO: count the training complaints per product with value_counts() on train["product"].

### Exercise 3: find complaints that mention a word

Choose one word that might signal a need for action, for example *refund*, *fraud* or *locked*. Use letters only.

1. Use `train["narrative"].str.contains("your word", case=False)` to find complaints containing these letters. `case=False` ignores capitalization, so *Refund* and *refund* both count. This also matches parts of longer words: *refund* also finds *refunded*.
2. Show how many training complaints contain it. Check the count first. If it is zero, choose another word and repeat the search before selecting a complaint.
3. Print the full narrative of one matching complaint, with its `complaint_id`, and read it.

In [ ]:
# Exercise 3: your code here
# TODO: mark the training complaints that contain your word (str.contains with case=False).
# TODO: print how many complaints contain it (.sum() counts the True values).
# TODO: if the count is 0, choose another word and run the cell again.
# TODO: keep the matching rows, pick one, and print its complaint_id and full narrative.

### Practice interpretation

Write 3–5 sentences:

- Choose two of the five complaints from 2.7 and give their `complaint_id`. For each, who at the company would act on it, and what decision would it support?
- In Exercise 3, how many training complaints contained your word? Did the complaint you read actually need the action your word suggests? What does that tell you about a simple keyword rule?
- Why do the product counts in Exercise 2 not tell you which problems are most common for consumers?

**Your answer:**



### Submit

1. Run the whole notebook from the top (**Restart**, then **Run All**) and check that there are no errors. Run All does not fill in your written answers, so check those too.
2. Save the notebook, so the outputs are saved in the file. The saved outputs are what is graded.
3. In GitHub Desktop, write a short summary such as "Lab 1 complete", click **Commit to main**, then **Push origin**.
4. Push before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

Discussion 1 is a separate Canvas item. Your decision note can help you prepare, but write the post itself in your own words.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**



## Part 3 Challenge (optional, not graded)

### A. Do the words match the product label?

The product label is the consumer's choice, so it is usually right but not always. This cell shows, for each product, the share of training complaints whose narrative contains the letters *card*, ignoring capitalization. This also matches parts of longer words, such as *cards* or *discard*. The mean of True/False values is a share, because True counts as 1 and False as 0. Then try it yourself: print one `Credit card` complaint that never mentions *card*. What is it about? Would a team that routes complaints by keyword miss it?

In [ ]:
# True for each training complaint whose narrative contains the letters "card" (case=False ignores capitals).
mentions_card = train["narrative"].str.contains("card", case=False)

# Group the True/False values by product; the mean in each group is a share.
mentions_card.groupby(train["product"]).mean().round(3)

### B. A project idea

Do you already have a business text source in mind? Write two sentences: what decision could it support, and what limits might you face with the data? No data collection is needed.

**Your idea:**

## Further reading

- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [pandas: working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/)

Next, Module 2 checks this feedback data for missing and repeated text, writes a review policy and keeps some messages aside for checking.